# Exploration, balance, and the experiment readout

The Criteo Uplift Prediction Dataset v2 (CC BY-NC-SA 4.0) is a collection of incrementality tests rebalanced to a constant assignment propensity of 0.85. The released data are a non-uniform subsample, so every level estimated here is a within-sample estimate, not Criteo's true incrementality. This notebook loads the seeded analysis sample saved by scripts/run_analysis.py and runs the packaged readout harness: ITT estimates, the SRM check, randomisation balance, and the exposure readout.

In [1]:
import sys
sys.path.insert(0, "../src")
import json
import pandas as pd
from uplift_platform.harness import balance_check, exposure_readout, itt_estimate, srm_check

pd.set_option("display.width", 140)
config = json.loads(open("../configs/default.json").read())
FEATURES = config["data"]["feature_columns"]
PROPENSITY = config["design"]["propensity"]


## The analysis sample

In [2]:
sample = pd.read_parquet('../data/processed/analysis_sample.parquet')
print('rows:', len(sample))
sample[['treatment', 'conversion', 'visit', 'exposure']].mean()

rows: 1000000


treatment     0.850918
conversion    0.002909
visit         0.046999
exposure      0.030870
dtype: float64

## Intention-to-treat estimates (difference in means)

In [3]:
for outcome in ('conversion', 'visit'):
    est = itt_estimate(sample[outcome].to_numpy(float), sample['treatment'].to_numpy(int))
    print(outcome, {k: est[k] for k in ('rate_treated', 'rate_control', 'ate', 'se', 'ci_low', 'ci_high')})

conversion {'rate_treated': 0.003096655611939106, 'rate_control': 0.0018379147046591808, 'ate': 0.0012587409072799252, 'se': 0.0001262282320038368, 'ci_low': 0.0010113381187202388, 'ci_high': 0.0015061436958396116}
visit {'rate_treated': 0.0486086790971633, 'rate_control': 0.03781140580351753, 'ate': 0.01079727329364577, 'se': 0.0005462491373899171, 'ci_low': 0.00972664465777546, 'ci_high': 0.01186790192951608}


## Sample ratio mismatch check against the designed 85/15 split

In [4]:
t = sample['treatment']
srm_check(int((t == 1).sum()), int((t == 0).sum()), PROPENSITY)

{'n_treated': 850918,
 'n_control': 149082,
 'observed_treated_share': 0.850918,
 'expected_treated_share': 0.85,
 'chi2': 6.609600000000355,
 'p_value': 0.010143044290033949,
 'alpha': 0.001,
 'verdict': 'no_srm_detected'}

## Randomisation balance: standardised mean differences

In [5]:
bal = balance_check(sample, FEATURES)
pd.DataFrame(bal['per_feature'])[['feature', 'smd', 'abs_smd']]

,feature,smd,abs_smd
0,f0,-0.010165,0.010165
1,f1,0.023872,0.023872
2,f2,-0.007811,0.007811
3,f3,-0.052646,0.052646
4,f4,0.006906,0.006906
5,f5,-0.026633,0.026633
6,f6,-0.039690,0.039690
7,f7,0.018784,0.018784
8,f8,-0.023144,0.023144
9,f9,0.024558,0.024558


In [6]:
print('max |SMD|:', bal['max_abs_smd'])
print('joint check:', bal['joint'])

max |SMD|: 0.05264600006573556
joint check: {'method': 'logistic likelihood-ratio test of assignment on all features', 'n_used': 200000, 'lr_statistic': 108.71764014768996, 'df': 12, 'p_value': 1.0727171969054081e-17}


## Exposure readout

Exposure is an attribute of the treated arm only in this data: it is 1 for a minority of treated rows and 0 for every control row. The harness therefore returns a descriptive contrast with an explicit warning, never a causal per-exposure effect.

In [7]:
exposure_readout(sample, 'conversion')

{'available': True,
 'label': 'descriptive_only_not_causal',
 'treated_exposure_share': 0.03627846631520311,
 'control_exposure_share': 0.0,
 'structural_zero_in_control': True,
 'warning': 'Exposure is structurally zero in the control arm: exposed users exist only among the treated, so no randomised exposed-versus-unexposed contrast exists. This figure is descriptive only and must not be read as the effect of exposure.',
 'n_exposed_treated': 30870,
 'rate_exposed_treated': 0.05442176870748299,
 'rate_control': 0.0018379147046591808,
 'descriptive_difference': 0.05258385400282381,
 'se': 0.0012958979646649528,
 'ci_low': 0.05004394066444174,
 'ci_high': 0.05512376734120588}